# RelFlow arena on Colab: Optuna loop for one RelArena task

Runs RelFlow on a RelBench task the way RelArena's harness runs a model, then searches
hyperparameters with Optuna on the **validation** score and refits every scored trial on
train + validation to record its **test** score, so the notebook shows how each configuration
converts. Nothing in the search reads test; test labels are only read inside RelBench's evaluator
during each refit.

**Before running:** Runtime → Change runtime type → a GPU (T4 works; L4 or A100 is faster).
Progress is mirrored to Google Drive every few minutes, so when the session ends, open the notebook
again and **Run all**: the loop resumes from its saved state and Optuna study. To stop cleanly,
create a file named `STOP` in the run directory (last cell shows the path).

In [ ]:
# Parameters
TASK = "rel-stack/post-votes"   # dataset/task from RelBench
TRIALS = 40                     # Optuna proposals beyond the default configuration
TRIAL_SECONDS = 3 * 3600        # wall-clock cap per trial
SESSION_HOURS = 11.5            # stop proposing new trials after this many hours (resume by running again)
DRIVE_DIR = "relflow-arena"     # folder under My Drive that keeps the run's state between sessions

In [ ]:
%pip install -q "relflow-arena[text] @ git+https://github.com/relflow/arena"
import importlib, sys
print("python", sys.version.split()[0]); importlib.import_module("benchmarks.relarena.sweep"); print("relflow-arena ready")

In [ ]:
import os, shutil, subprocess, sys, time, threading
from pathlib import Path

NAME = TASK.replace("/", "-")
RUN = Path(f"/content/runs/{NAME}")            # fast local disk for the loop
CACHE = Path("/content/cache/relbench")
RUN.mkdir(parents=True, exist_ok=True); CACHE.mkdir(parents=True, exist_ok=True)
os.environ.update(RELBENCH_CACHE_DIR=str(CACHE), HF_HOME="/content/cache/huggingface", PYTHONUNBUFFERED="1")

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    MIRROR = Path("/content/drive/MyDrive") / DRIVE_DIR / NAME
except Exception:
    MIRROR = Path("/content/mirror") / NAME   # not on Colab: mirror locally
MIRROR.mkdir(parents=True, exist_ok=True)

KEEP = ("state.json", "optuna.db", "plan.json", "README.md", "sweep.log", "launch.log")

def sync(src: Path, dst: Path) -> int:
    """Copy the loop's small, resumable files (state, study, plan, per-trial results) from src to dst."""
    copied = 0
    for name in KEEP:
        if (src / name).exists():
            shutil.copyfile(src / name, dst / name); copied += 1
    for result in list(src.glob("trial-*/result.json")) + list(src.glob("trial-*/refit/result.json")) + list(src.glob("refit/result.json")):
        target = dst / result.relative_to(src)
        target.parent.mkdir(parents=True, exist_ok=True); shutil.copyfile(result, target); copied += 1
    for derived in src.glob("derived/**/*"):
        if derived.is_file():
            target = dst / derived.relative_to(src); target.parent.mkdir(parents=True, exist_ok=True); shutil.copyfile(derived, target)
    return copied

restored = sync(MIRROR, RUN) if (MIRROR / "state.json").exists() else 0
print("run directory", RUN, "| mirror", MIRROR, "| restored files:", restored)

In [ ]:
from relbench.datasets import get_dataset
from relbench.tasks import get_task

dataset_name, task_name = TASK.split("/")
dataset = get_dataset(dataset_name, download=True)
task = get_task(dataset_name, task_name, download=True)
DIRECTION = "minimize" if str(task.task_type).lower().endswith("regression") else "maximize"
print(TASK, "| task type", task.task_type, "| optimize direction", DIRECTION)
print("train / val / test rows:", *(len(task.get_table(split).df) for split in ("train", "val", "test")))

In [ ]:
if not (RUN / "plan.json").exists():
    subprocess.run([sys.executable, "-m", "benchmarks.relarena.sweep", "plan", "--task", TASK, "--cache", str(CACHE),
                    "--trials", "0", "--seed", "0", "--output", str(RUN / "plan.json")], check=True)
print((RUN / "plan.json").read_text()[:400])

In [ ]:
stop_sync = threading.Event()

def mirror_loop():
    while not stop_sync.wait(300):
        try:
            sync(RUN, MIRROR)
        except Exception as error:  # Drive hiccups must not stop the loop
            print("mirror:", error)

threading.Thread(target=mirror_loop, daemon=True).start()
print("mirroring", RUN, "->", MIRROR, "every 5 minutes")

In [ ]:
argv = [
    sys.executable, "-m", "benchmarks.relarena.sweep", "optimize",
    "--plan", str(RUN / "plan.json"), "--name", f"{NAME}-colab", "--output", str(RUN),
    "--trials", str(TRIALS), "--startup", "8", "--direction", DIRECTION,
    "--python", sys.executable, "--source", str(RUN), "--cache", str(CACHE), "--seed", "0",
    "--trial-seconds", str(TRIAL_SECONDS), "--max-hours", str(SESSION_HOURS),
    "--refit-every-trial", "--num-workers", "auto",
]
print(" ".join(argv))
with open(RUN / "launch.log", "a") as log:
    log.write(f"started {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
process = subprocess.Popen(argv, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
with open(RUN / "sweep.log", "a") as log:
    for line in process.stdout:
        log.write(line)
        if line.startswith("{"):
            print(line.rstrip()[:240])
print("exit", process.wait())

In [ ]:
stop_sync.set(); print("final mirror copied", sync(RUN, MIRROR), "files to", MIRROR)
import json
state = json.loads((RUN / "state.json").read_text())
metric = next((t["summary"]["metric"] for t in state["trials"] if t.get("summary", {}).get("metric")), "?")
higher = metric == "roc_auc"
done = sorted([t for t in state["trials"] if t.get("status") == "done" and t.get("summary", {}).get("score") is not None],
              key=lambda t: t["summary"]["score"], reverse=higher)
print(f"{TASK}: {len(done)} scored trials; {metric} ({'higher' if higher else 'lower'} is better)")
print("| val rank | trial | validation | test | dropout | lr | wd | ema | jitter | mask | loss |")
print("|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|")
for rank, t in enumerate(done, 1):
    c, s = t["config"], t["summary"]; test = ((t.get("refit") or {}).get("test") or {}).get(metric)
    print(f"| {rank} | {t['index']} | {s['score']:.5f} | {'' if test is None else f'{test:.5f}'} | {c['dropout']} | {c['learning_rate']:.1e} | {c['weight_decay']} | {c['ema_decay'] or '-'} | {c.get('jitter', 0)} | {c.get('mask', 0)} | {c.get('objective', 'mae')} |")
print("\nTo stop the loop cleanly next session: create", RUN / "STOP")